# 22 — Replication & Consistency

### Why this exists
Replication creates copies; consistency defines what readers are allowed to observe when those copies diverge. This lab simulates lag and read-after-write behavior.

### Learning objectives
model leader/follower state; inject lag; compare read policies; measure stale reads

### Prerequisite
Python lists and simple state

### Engineering loop
**Predict → Build → Observe → Break → Diagnose → Improve → Generalize → Defend**

## 🖊️ Sketch note
![Sketch note](../assets/sketch-notes/22-replication-consistency.svg)

> **Question to carry through the lab:** When a user writes an order and immediately reads it, which replica is safe to use?

## Capability contract

**Capability:** C05 — Storage Consistency & Distributed Systems  
**Workstream:** Distributed Data Platform  
**Primary roles:** Data Engineer|Platform Engineer|Architect  
**You will prove:** Explain a distributed trade-off


## Mental model
Replication and consistency are separate decisions. A follower can be healthy yet stale. Stronger read guarantees often cost latency, availability, or routing complexity.

## 1. Predict before you run
Write down what you expect and what evidence would prove you wrong. Do not change the experiment after seeing the result.

## 2. Build
The first cell creates the smallest system that can answer the question. The later cells change one boundary at a time.

## Agent-native lens

**Agent can do:** routine implementation or a first-pass analysis.

**You must understand:** the mechanism and its assumptions.

**You must verify:** use tests, measurements or adversarial data that could prove the output wrong.

**You must decide:** the trade-off, failure response or production boundary.


## Apprenticeship bridge

Use this notebook to prepare for the following Acme Commerce work scenario(s). The scenario gives you symptoms and constraints; it does **not** give you the diagnosis.

- **S08 — Replication and consistency (DATA-1110)** → [`../work_simulations/08_acme_commerce_replica_lag/README.md`](../work_simulations/08_acme_commerce_replica_lag/README.md)

**How to use the bridge:** finish the experiment and independent challenge here, then enter the ticket without reopening the notebook as an answer key. Bring your own prediction, evidence and verification plan.


In [ ]:
leader=[]; follower=[]
def write(event): leader.append(event)
def replicate(n=1): follower.extend(leader[len(follower):len(follower)+n])
def read(replica): return replica[-1] if replica else None
write("order=o1")
replicate(0)
print({"leader":read(leader),"follower":read(follower)})

In [ ]:
# Measure lag as events not yet applied.
print("lag",len(leader)-len(follower))
replicate(1)
print("after catch-up",read(follower),"lag",len(leader)-len(follower))

In [ ]:
write("order=o2")
print("stale follower read",read(follower))
print("leader read",read(leader))

In [ ]:
def read_after_write(preferred, fallback, required_index):
    if len(preferred)>=required_index: return preferred[-1] if preferred else None,"preferred"
    if len(fallback)>=required_index: return fallback[-1],"fallback"
    return None,"wait"
print(read_after_write(follower,leader,len(leader)))

## 3. Measure
Do not stop at “it worked”. Record a correctness signal, a failure signal, and at least one quantitative measurement where the concept permits it.

## 4. Break deliberately
Introduce one controlled failure. Keep the failure reproducible so another engineer can reproduce the observation.

## 5. Diagnose
Use evidence, not the implementation you expected. State: **symptom → mechanism → evidence → boundary**. If two explanations fit, design one more measurement.

## 6. Improve
Make the smallest change that addresses the mechanism. Re-run the original experiment and the failure case.

## 7. Generalize
Ask what changes at 10× load, with retries, multiple writers/readers, partial failure, or changing contracts. Separate what this toy proves from what it only illustrates.

## Your task

**Goal:** Choose a read-after-write policy under replica lag.

**Do this:** Compare leader reads, session stickiness, quorum-style reads and waiting for catch-up. Quantify the latency/availability trade-off.

**Before you finish, save:** your prediction, the key measurement, the failure/edge case you tested, your diagnosis, and the production implication.

**Done when:** another engineer can reproduce your result and understand why you made the decision.

### Exercise detail

## 8. Production bridge
Production replication requires replication topology, failure handling, consistency guarantees, lag metrics, routing policy and explicit business tolerance for stale data.

## 9. Independent challenge
Compare leader reads, session stickiness, quorum-style reads, and waiting for replica catch-up. Quantify the latency/availability trade-off you are assuming.

### Evidence to keep
lag measurement, stale read reproduction, and a written read-after-write policy

## 10. Explain it in 60–90 seconds
Explain the mechanism, your measurement, the failure you injected, the diagnosis, and the production decision you would make. Avoid tool names until the mental model is clear.

---

**Check your work:** [Open the reference solution](https://github.com/ankit-rathi/data-systems-lab/blob/main/solutions/notebooks/22.md) — only after you have attempted the exercise.